# NB07 — TCGA 31-Class Pan-Cancer Evaluation

Loads OpenSlideFM slide embeddings produced by NB06, aggregates to patient level by mean pooling, and runs 5-fold stratified group cross-validation across 3 random seeds (15 evaluations total) using TSS codes as grouping variable. Reports accuracy, balanced accuracy, macro-F1, macro-AUROC, with bootstrap 95% CIs. Saves OOF arrays for downstream figure rendering in NB13.

In [ ]:
import os, json, datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
)
from scipy.stats import ttest_rel
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))

class CFG:
    EMBEDDINGS_DIR = WORKSPACE / 'embeddings'
    LABELS = WSI_ROOT.parent / 'artifacts' / 'labels' / 'labels.csv'
    MANIFEST = WORKSPACE / 'manifests' / 'manifest_tcga.csv'
    OUTPUT = WORKSPACE / 'results' / 'tcga_baseline_evaluation'
    SEEDS = [42, 123, 456]
    N_FOLDS = 5
    BOOTSTRAP_N = 1000

CFG.OUTPUT.mkdir(parents=True, exist_ok=True)

def load_openslidefm_embeddings():
    print('[LOAD] reading OpenSlideFM embeddings')
    records = {}
    def _try_add(npy_path):
        sid = npy_path.stem
        if sid in records: return
        try:
            emb = np.load(npy_path).astype(np.float32)
        except Exception:
            return
        if emb.ndim == 1 and emb.shape[0] == 768:
            records[sid] = emb
    if CFG.EMBEDDINGS_DIR.exists():
        for sub in CFG.EMBEDDINGS_DIR.iterdir():
            if sub.is_dir():
                for npy_path in sub.glob('*.npy'):
                    _try_add(npy_path)
        for npy_path in CFG.EMBEDDINGS_DIR.glob('*.npy'):
            _try_add(npy_path)
        sf_dir = CFG.EMBEDDINGS_DIR / 'student_final'
        if sf_dir.exists():
            for npy_path in sf_dir.glob('*.npy'):
                _try_add(npy_path)
    print(f'[LOAD] found {len(records)} unique slide embeddings')
    import re
    patient_embs = {}
    for sid, emb in records.items():
        m = re.search(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', sid)
        if m:
            pid = m.group(1)
            patient_embs.setdefault(pid, []).append(emb)
    rows = []
    for pid, embs in patient_embs.items():
        mean_emb = np.mean(embs, axis=0)
        row = {'patient_id': pid}
        for i in range(768):
            row[f'f{i}'] = float(mean_emb[i])
        rows.append(row)
    df = pd.DataFrame(rows).set_index('patient_id')
    print(f'[LOAD] aggregated to {len(df)} patients')
    return df

def prepare_dataset(df_emb, df_manifest):
    df_manifest['patient_id'] = df_manifest['slide_id'].str.extract(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', expand=False)
    patient_cancer_map = df_manifest.groupby('patient_id')['cancer_code'].first().to_dict()
    df_manifest['tss'] = df_manifest['patient_id'].str.extract(r'TCGA-([A-Z0-9]{2})-', expand=False)
    patient_tss_map = df_manifest.groupby('patient_id')['tss'].first().to_dict()
    df_emb = df_emb.copy()
    df_emb['patient_id'] = df_emb.index
    df_emb['cancer_type'] = df_emb['patient_id'].map(patient_cancer_map)
    df_emb['tss'] = df_emb['patient_id'].map(patient_tss_map)
    df_emb = df_emb[df_emb['cancer_type'].notna() & df_emb['tss'].notna()].copy()
    print(f'  patients with labels + tss: {len(df_emb)}')
    feature_cols = [c for c in df_emb.columns if c.startswith('f')]
    X = df_emb[feature_cols].values.astype(np.float32)
    le = LabelEncoder()
    y = le.fit_transform(df_emb['cancer_type'].values)
    tss = df_emb['tss'].values
    print(f'  feature matrix: {X.shape}')
    print(f'  number of classes: {len(le.classes_)}')
    return X, y, tss, le, df_emb

def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=42, metric_fn=accuracy_score):
    rng = np.random.default_rng(seed)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            pass
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

print('TCGA 31-class evaluation: LogisticRegression, TSS-grouped CV, 3 seeds')
df_emb = load_openslidefm_embeddings()
df_manifest = pd.read_csv(CFG.MANIFEST)
X, y, tss, le, df_labeled = prepare_dataset(df_emb, df_manifest)

all_fold_results = []
oof_preds = np.zeros(len(y), dtype=int)
oof_probs = np.zeros((len(y), len(le.classes_)), dtype=np.float32)
oof_counts = np.zeros(len(y), dtype=int)

for seed in CFG.SEEDS:
    print(f'\nseed {seed}')
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=seed)
    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=tss)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        clf = LogisticRegression(C=1.0, penalty='l2', solver='saga',
                                 max_iter=5000, random_state=seed, n_jobs=-1)
        clf.fit(X_train_s, y_train)
        y_pred = clf.predict(X_test_s)
        y_probs = clf.predict_proba(X_test_s)
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        try:
            auc = roc_auc_score(y_test, y_probs, multi_class='ovr', average='macro')
        except Exception:
            auc = None
        all_fold_results.append({
            'seed': seed, 'fold': fold_idx, 'accuracy': acc,
            'balanced_accuracy': bal_acc, 'macro_f1': macro_f1, 'macro_auc': auc,
        })
        oof_preds[test_idx] = y_pred
        oof_probs[test_idx] = y_probs
        oof_counts[test_idx] += 1
        print(f'  fold {fold_idx+1}/{CFG.N_FOLDS}: acc={acc:.4f} bal-acc={bal_acc:.4f} f1={macro_f1:.4f}')

df_results = pd.DataFrame(all_fold_results)
mean_acc = df_results['accuracy'].mean(); std_acc = df_results['accuracy'].std()
mean_bal = df_results['balanced_accuracy'].mean(); std_bal = df_results['balanced_accuracy'].std()
mean_f1 = df_results['macro_f1'].mean(); std_f1 = df_results['macro_f1'].std()
auc_vals = df_results['macro_auc'].dropna()
mean_auc = auc_vals.mean() if len(auc_vals) else None
std_auc = auc_vals.std() if len(auc_vals) else None
ci_acc = bootstrap_ci(y, oof_preds, n_boot=CFG.BOOTSTRAP_N, metric_fn=accuracy_score)

print(f'\nresults across {len(df_results)} evaluations:')
print(f'  accuracy:          {mean_acc:.4f} +/- {std_acc:.4f}  95% CI: [{ci_acc[0]:.4f}, {ci_acc[1]:.4f}]')
print(f'  balanced accuracy: {mean_bal:.4f} +/- {std_bal:.4f}')
print(f'  macro F1:          {mean_f1:.4f} +/- {std_f1:.4f}')
if mean_auc is not None:
    print(f'  macro AUROC (OvR): {mean_auc:.4f} +/- {std_auc:.4f}')

results = {
    'timestamp': datetime.datetime.now().isoformat(),
    'method': 'LogisticRegression_C1.0_L2',
    'cv': 'StratifiedGroupKFold_TSS',
    'seeds': CFG.SEEDS,
    'n_evaluations': len(df_results),
    'accuracy': {'mean': float(mean_acc), 'std': float(std_acc), 'ci95': list(ci_acc)},
    'balanced_accuracy': {'mean': float(mean_bal), 'std': float(std_bal)},
    'macro_f1': {'mean': float(mean_f1), 'std': float(std_f1)},
    'macro_auc_ovr': {'mean': float(mean_auc), 'std': float(std_auc)} if mean_auc else None,
}
with open(str(CFG.OUTPUT / 'test_metrics.json'), 'w') as f:
    json.dump(results, f, indent=2)
df_results.to_csv(str(CFG.OUTPUT / 'fold_results.csv'), index=False)

report = classification_report(y, oof_preds, target_names=le.classes_, output_dict=True)
pd.DataFrame(report).T.to_csv(str(CFG.OUTPUT / 'per_class_metrics.csv'))

cm = confusion_matrix(y, oof_preds)
pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_csv(str(CFG.OUTPUT / 'confusion_matrix.csv'))

np.savez(str(CFG.OUTPUT / 'oof.npz'),
         y=y, oof_preds=oof_preds, oof_probs=oof_probs,
         classes=np.array(le.classes_, dtype=object))

print(f'\n[OK] results saved to: {CFG.OUTPUT}')
print('NB07 complete. Next: NB08 (embeddings export).')